# Analysis of bond prices of South Africa (Africa), India (developing economies) and the US (developed markets) from 2022 to 2024
This notebook assembles the full analysis: data loading, descriptive statistics, trend plots, moving averages, volatility, correlation and connectedness analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.api import VAR

df = pd.read_csv('bond_format2.csv')
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m')
df = df.sort_values('Date')
df['Return'] = df.groupby('Country')['Clean_Price'].pct_change()
df.info()
df.head()

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'statsmodels'

## Descriptive Statistics
This is the summary statistics for each sovereign bond market: Clean Price, Yield to Maturity, Coupon Rate and the monthly return.

In [ ]:
countries = df['Country'].unique().tolist()
descriptions = {c: df[df['Country'] == c][['Clean_Price','Yield_to_Maturity','Coupon_Rate','Return']].describe() for c in countries}
descriptions

## Price and Yield Trends
I plotted the price and yield time series of each country to visualize their evolution over the observed period (2020-2022).

In [ ]:
fig, axes = plt.subplots(2,1, figsize=(10,8), sharex=True)
for c in countries:
    sub = df[df['Country'] == c]
    axes[0].plot(sub['Date'], sub['Clean_Price'], label=c)
    axes[1].plot(sub['Date'], sub['Yield_to_Maturity'], label=c)
axes[0].set_title('Clean Price Trends')
axes[0].set_ylabel('Price')
axes[1].set_title('Yield to Maturity Trends')
axes[1].set_ylabel('Yield (%)')
axes[1].set_xlabel('Date')
for ax in axes:
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()

## Moving Averages
Compute and plot the 3-month and 6-month moving averages of Clean Price for each country.

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
for c in countries:
    sub = df[df['Country'] == c].set_index('Date').sort_index()
    ma3 = sub['Clean_Price'].rolling(window=3).mean()
    ma6 = sub['Clean_Price'].rolling(window=6).mean()
    ax.plot(sub.index, ma3, label=f'{c} MA3')
    ax.plot(sub.index, ma6, label=f'{c} MA6', linestyle='--')
ax.set_title('3- and 6-Month Moving Averages of Clean Price')
ax.set_xlabel('Date')
ax.set_ylabel('Price')
ax.legend(fontsize='small')
ax.grid(True)
plt.tight_layout()
plt.show()

## Volatility Dynamics
Compute and plot rolling 3- and 6-month standard deviation of returns as a measure of volatility for each country.

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
for c in countries:
    sub = df[df['Country'] == c].set_index('Date').sort_index()
    vol3 = sub['Return'].rolling(window=3).std()
    vol6 = sub['Return'].rolling(window=6).std()
    ax.plot(sub.index, vol3, label=f'{c} Vol3')
    ax.plot(sub.index, vol6, label=f'{c} Vol6', linestyle='--')
ax.set_title('Rolling 3- and 6-Month Volatility of Returns')
ax.set_xlabel('Date')
ax.set_ylabel('Volatility (std)')
ax.legend(fontsize='small')
ax.grid(True)
plt.tight_layout()
plt.show()

## Correlation Analysis
Build a pivot table of monthly returns with countries as columns, compute the correlation matrix, and visualize it.

In [ ]:
returns = df.pivot(index='Date', columns='Country', values='Return').dropna()
corr_matrix = returns.corr()
corr_matrix

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Returns')
plt.tight_layout()
plt.show()

## Connectedness Analysis via VAR
Fit a VAR to the return series, select appropriate lag via AIC (default up to 4 lags), and compute the 10-step ahead forecast error variance decomposition (FEVD).

In [ ]:
model = VAR(returns)
aic = model.select_order(maxlags=4)
lag_order = aic.aic
if np.isnan(lag_order) or lag_order < 1:
    lag_order = 1
else:
    lag_order = int(lag_order)
print(f'Selected lag order: {lag_order}')
res = model.fit(lag_order)
print(res.summary())
H = 10
fevd = res.fevd(H)
fevd_matrix = np.array([fevd.decomp[i, H-1, :] for i in range(len(returns.columns))])
print('FEVD matrix at horizon %d:' % H)
print(fevd_matrix)
labels = returns.columns.tolist()
fevd_df = pd.DataFrame(fevd_matrix, index=labels, columns=labels)
fevd_df

In [ ]:
n = len(labels)
off_sum = fevd_matrix.sum() - np.trace(fevd_matrix)
TCI = 100 * off_sum / n
dir_to = fevd_matrix.sum(axis=1) - np.diag(fevd_matrix)
dir_from = fevd_matrix.sum(axis=0) - np.diag(fevd_matrix)
print(f'Total Connectedness Index (TCI) = {TCI:.2f}%')
print('Directional TO others:', dict(zip(labels, dir_to)))
print('Directional FROM others:', dict(zip(labels, dir_from)))
plt.figure(figsize=(6,4))
bottom = np.zeros(n)
colors = ['C0','C1','C2']
for j in range(n):
    plt.bar(labels, fevd_matrix[:, j], bottom=bottom, color=colors[j], label=labels[j])
    bottom += fevd_matrix[:, j]
plt.ylabel('FEVD share')
plt.title(f'Stacked FEVD shares at horizon {H}')
plt.legend(title='Shock origin', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.show()